In [2]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import soundfile as sf

# ==============================================================================
# 1. GPU ENABLED FBANK EXTRACTOR
# ==============================================================================
class FBankExtractor:
    def __init__(self, device, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160):
        self.sample_rate = sample_rate
        self.device = device

        # Khởi tạo transform MelSpectrogram trực tiếp trên GPU
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, 
            n_fft=n_fft, 
            n_mels=n_mels, 
            hop_length=hop_length, 
            center=False
        ).to(self.device)

    def _apply_cmvn(self, feature):
        """Áp dụng chuẩn hóa Mean and Variance Normalization"""
        mean = feature.mean(dim=-1, keepdim=True)
        std = feature.std(dim=-1, keepdim=True)
        return (feature - mean) / (std + 1e-6)

    def extract(self, waveform_cpu):
        # Đưa dữ liệu lên GPU
        waveform_gpu = waveform_cpu.to(self.device)
        if waveform_gpu.dim() == 1: 
            waveform_gpu = waveform_gpu.unsqueeze(0)

        # Tính toán FBank (Log-Mel Spectrogram)
        mel_spec = self.mel_transform(waveform_gpu)
        fbank = torch.log(mel_spec + 1e-6)

        # Chuẩn hóa
        fbank = self._apply_cmvn(fbank)
        
        return fbank.squeeze(0) # Trả về tensor (n_mels, time)

# ==============================================================================
# 2. DATASET (LOGIC RESUME & SCAN)
# ==============================================================================
class FBankDataset(Dataset):
    def __init__(self, root_dir, output_dir, extractor, sample_rate=16000):
        self.root_dir = root_dir
        self.output_dir = output_dir
        self.extractor = extractor
        self.sample_rate = sample_rate
        self.file_list = []
        
        print(f"-> Đang quét thư mục đầu vào: {root_dir}")
        
        # Quét tệp và kiểm tra Resume logic
        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.wav', '.flac', '.mp3')):
                    input_path = os.path.join(root, file)
                    
                    # Tính toán đường dẫn file .pt sẽ lưu
                    rel_path = os.path.relpath(input_path, root_dir)
                    pt_filename = os.path.splitext(rel_path)[0] + ".pt"
                    expected_path = os.path.join(output_dir, pt_filename)

                    if not os.path.exists(expected_path):
                        self.file_list.append(input_path)

        print(f"-> Tổng số tệp mới cần xử lý: {len(self.file_list)}")

    def __len__(self): 
        return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]
        try:
            wav_numpy, sr = sf.read(path)
            waveform = torch.from_numpy(wav_numpy).float()
            
            # Chuẩn hóa channel
            if waveform.dim() == 1: 
                waveform = waveform.unsqueeze(0)
            else: 
                waveform = waveform.t()
            
            if waveform.shape[0] > 1: 
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            # Resample nếu cần
            if sr != self.sample_rate:
                resampler = T.Resample(sr, self.sample_rate)
                waveform = resampler(waveform)

            # Trích xuất FBank (trả về tensor trên GPU hoặc CPU tùy thiết lập)
            fbank = self.extractor.extract(waveform)
            
            return fbank.cpu(), path 
        except Exception as e:
            print(f" Lỗi xử lý {path}: {e}")
            return None, None

# ==============================================================================
# 3. EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # --- CẤU HÌNH ---
    INPUT_PATH = r"D:\Study\7-SP26\DATxSLP\Data_after_cut\test_output"
    OUTPUT_PATH = r"D:\Study\7-SP26\DATxSLP\Data_after_extract_feature\FBank_Only"
    BATCH_SIZE = 1 # Tăng giảm tùy vào VRAM của bạn
    
    # --- THIẾT BỊ ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔥 Đang chạy trên: {device}")

    extractor = FBankExtractor(device=device)
    dataset = FBankDataset(INPUT_PATH, OUTPUT_PATH, extractor)

    if len(dataset) == 0:
        print("\n✅ Tất cả tệp đã được xử lý xong!")
        exit()

    # Dùng num_workers=0 khi trích xuất audio trên Windows để tránh lỗi multiprocessing
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"\nBắt đầu trích xuất...")
    os.makedirs(OUTPUT_PATH, exist_ok=True)

    for batch in tqdm(loader):
        fbanks, paths = batch
        if fbanks is None: continue

        for i in range(len(paths)):
            # Tạo đường dẫn lưu trữ giữ nguyên cấu trúc thư mục gốc
            rel_path = os.path.relpath(paths[i], INPUT_PATH)
            save_path = os.path.join(OUTPUT_PATH, os.path.splitext(rel_path)[0] + ".pt")
            
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            
            # Lưu tensor FBank (n_mels, time)
            torch.save(fbanks[i], save_path)

    print(f"\n Hoàn thành! Dữ liệu lưu tại: {OUTPUT_PATH}")

🔥 Đang chạy trên: cuda
-> Đang quét thư mục đầu vào: D:\Study\7-SP26\DATxSLP\Data_after_cut\test_output
-> Tổng số tệp mới cần xử lý: 639

Bắt đầu trích xuất...


100%|██████████| 639/639 [00:09<00:00, 68.35it/s]


 Hoàn thành! Dữ liệu lưu tại: D:\Study\7-SP26\DATxSLP\Data_after_extract_feature\FBank_Only
